In [5]:
import cv2
import csv
import numpy as np
from pathlib import Path
from PIL import Image
from rembg import remove

# ======================
# USER INPUTS
# ======================

VIDEO_FOLDER = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/videos/")
CAMERA_FILENAME = "Cam2012631.mp4"
PREDICTION_CSV = Path("/mnt/lemebel/happyhouse_102025/session6/2025_10_12_15_06_46/videos/predictions/2026_01_20_16_45_12/Cam2012861.csv")

START_FRAME = 15132
END_FRAME = 15172
FRAME_INTERVAL = 1
CROP_MARGIN = 60
MIN_CROP_SIZE = 160

OUTPUT_ROOT = Path("/mnt/lemebel/frame_exports/one_step")

# ======================
# LOAD PREDICTIONS
# ======================

def load_predictions(csv_path):
    preds = {}
    with csv_path.open() as f:
        reader = csv.reader(f)
        next(reader)  # skeleton path
        for row in reader:
            frame = int(row[0])
            n_kp = (len(row)-1)//3
            pts = np.full((n_kp,2), np.nan)
            for i in range(n_kp):
                b = 1+3*i
                x,y = float(row[b+1]), float(row[b+2])
                if abs(x)>1e6 or abs(y)>1e6:
                    continue
                pts[i] = [x,y]
            preds[frame] = pts
    return preds

preds = load_predictions(PREDICTION_CSV)

# ======================
# SETUP OUTPUT DIRS
# ======================

camera_name = Path(CAMERA_FILENAME).stem
export_name = f"{camera_name}_frames_{START_FRAME}_{END_FRAME}_every_{FRAME_INTERVAL}_fullraw_nobg"
raw_dir = OUTPUT_ROOT / export_name / "raw_full"
clean_dir = OUTPUT_ROOT / export_name / "raw_full_nobg"

raw_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)

video_path = VIDEO_FOLDER / CAMERA_FILENAME
assert video_path.exists(), f"Video not found: {video_path}"

# ======================
# EXTRACT, REMOVE BG, PRESERVE FULL GEOMETRY
# ======================

cap = cv2.VideoCapture(str(video_path))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video frames: {total_frames}")

current_frame = START_FRAME
saved_count = 0

print("Processing frames (full raw + full no-bg)...")

while current_frame < END_FRAME:
    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
    ret, frame = cap.read()
    if not ret:
        print(f"Failed to read frame {current_frame}")
        break

    if current_frame not in preds:
        print(f"No prediction for frame {current_frame}, skipping.")
        current_frame += FRAME_INTERVAL
        continue

    pts = preds[current_frame]
    valid = ~np.isnan(pts[:,0])

    if not np.any(valid):
        print(f"No valid keypoints for frame {current_frame}, skipping.")
        current_frame += FRAME_INTERVAL
        continue

    xs = pts[valid,0]
    ys = pts[valid,1]

    # Compute permissive bounding box around fly
    xmin = int(xs.min() - CROP_MARGIN*1.5)
    xmax = int(xs.max() + CROP_MARGIN*1.5)
    ymin = int(ys.min() - CROP_MARGIN*1.5)
    ymax = int(ys.max() + CROP_MARGIN*1.5)

    h,w = frame.shape[:2]

    # Enforce minimum size
    cx = (xmin + xmax)//2
    cy = (ymin + ymax)//2
    half = max((xmax-xmin)//2, (ymax-ymin)//2, MIN_CROP_SIZE//2)

    xmin = cx - half
    xmax = cx + half
    ymin = cy - half
    ymax = cy + half

    # Clamp bounds
    xmin = max(0,xmin); ymin = max(0,ymin)
    xmax = min(w,xmax); ymax = min(h,ymax)

    crop = frame[ymin:ymax, xmin:xmax]

    # Save FULL raw frame (unchanged)
    fname = f"{camera_name}_frame_{current_frame:06d}.png"
    raw_path = raw_dir / fname
    cv2.imwrite(str(raw_path), frame)

    # ----- BACKGROUND REMOVAL ON CROP -----
    pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    rgba_crop = remove(pil_crop)

    # Extract alpha mask
    alpha = np.array(rgba_crop.split()[-1])

    # Preserve original colors
    crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    rgba_fixed = np.dstack([crop_rgb, alpha])

    # ----- REINSERT INTO FULL FRAME -----
    full_rgba = np.zeros((h, w, 4), dtype=np.uint8)
    full_rgba[..., 3] = 0  # fully transparent background

    full_rgba[ymin:ymax, xmin:xmax, :] = rgba_fixed

    # Save full-frame RGBA aligned with original camera geometry
    clean_path = clean_dir / fname
    Image.fromarray(full_rgba).save(clean_path)

    saved_count += 1
    current_frame += FRAME_INTERVAL

cap.release()
print(f"\nSaved {saved_count} frames to:\n{raw_dir}\n{clean_dir}")


Video frames: 1714786
Processing frames (full raw + full no-bg)...


2026-01-27 15:05:46.533634781 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcudnn.so.9: cannot open shared object file: No such file or directory

2026-01-27 15:05:47.752085688 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onnxruntime::Provider& onnxruntime::ProviderLibrary::Get() [ONNXRuntimeError] : 1 : FAIL : Failed to load library libonnxruntime_providers_cuda.so with error: libcudnn.so.9: cannot open shared object file: No such file or directory

2026-01-27 15:05:48.925945658 [E:onnxruntime:Default, provider_bridge_ort.cc:2251 TryGetProviderInfo_CUDA] /onnxruntime_src/onnxruntime/core/session/provider_bridge_ort.cc:1844 onn


Saved 40 frames to:
/mnt/lemebel/frame_exports/one_step/Cam2012631_frames_15132_15172_every_1_fullraw_nobg/raw_full
/mnt/lemebel/frame_exports/one_step/Cam2012631_frames_15132_15172_every_1_fullraw_nobg/raw_full_nobg


In [3]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from pathlib import Path
import numpy as np
from PIL import Image
import csv
import json

# ======================
# USER CONFIG
# ======================

GHOST_IMAGE_DIR = Path("/mnt/lemebel/frame_exports/Cam2012853_frames_10000_11000_every_20_fullraw_nobg/raw_full_nobg")
PREDICTION_CSV = Path("/mnt/lemebel/happyhouse_102025/session5/2025_10_11_10_29_50/videos/predictions/Cam2012853.csv")
OUTPUT_PATH = Path("/mnt/lemebel/animated_ghosts/Cam2012853_ghost.mp4")

FRAME_INTERVAL = 20   # must match extraction interval
ALPHA_DECAY = 0.85    # how fast ghosts fade
FPS = 10              # video speed

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ======================
# Load trajectory
# ======================

def load_keypoints2d(csv_path):
    with csv_path.open() as f:
        reader = csv.reader(f)
        skel_path = Path(next(reader)[0])
        if not skel_path.exists():
            skel_path = csv_path.parent/skel_path
        node_names = json.load(open(skel_path))["node_names"]
        rows = list(reader)

    frames, pts = [], []
    n_kp = (len(rows[0])-1)//3
    for row in rows:
        frames.append(int(row[0]))
        kp = np.full((n_kp,2), np.nan)
        for i in range(n_kp):
            b=1+3*i
            x,y=float(row[b+1]),float(row[b+2])
            if abs(x)>1e6 or abs(y)>1e6:
                continue
            kp[i]=[x,y]
        pts.append(kp)
    return np.array(frames), np.array(pts), node_names

frames, points, node_names = load_keypoints2d(PREDICTION_CSV)

# ======================
# Load ghost frames
# ======================

ghost_files = sorted(GHOST_IMAGE_DIR.glob("*.png"))
assert ghost_files, "No ghost images found!"

imgs = [np.array(Image.open(p)) for p in ghost_files]
h,w,_ = imgs[0].shape

# ======================
# Build animation
# ======================

fig, ax = plt.subplots(figsize=(6,6))
ax.set_xlim(0,w)
ax.set_ylim(h,0)
ax.axis("off")

ghost_stack = []

traj_line, = ax.plot([], [], lw=2, color="cyan")

def update(i):
    ax.clear()
    ax.set_xlim(0,w)
    ax.set_ylim(h,0)
    ax.axis("off")

    # Add ghosts (fading)
    for j,img in enumerate(ghost_stack):
        alpha = (ALPHA_DECAY**(len(ghost_stack)-j-1))
        ax.imshow(img, alpha=alpha)

    # Add current frame
    ax.imshow(imgs[i], alpha=1.0)
    ghost_stack.append(imgs[i])

    # Grow trajectory
    frame_num = int(ghost_files[i].stem.split("_")[-1])
    mask = frames <= frame_num
    traj = points[mask,:, :]

    x = traj[:,0,0]  # just kp 0 for demo
    y = traj[:,0,1]
    valid = ~np.isnan(x)&~np.isnan(y)
    ax.plot(x[valid], y[valid], color="yellow", lw=1.5)

    return []

ani = animation.FuncAnimation(fig, update, frames=len(imgs), interval=1000/FPS)

print("Saving animation...")
ani.save(OUTPUT_PATH, writer="ffmpeg", fps=FPS)
plt.close(fig)

print(f"Saved animated ghost to:\n{OUTPUT_PATH}")


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/lemebel/happyhouse_102025/session5/2025_10_11_10_29_50/videos/predictions/Cam2012853.csv'